In [1]:
cell_str=r'''
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>
#include <cublas_v2.h> // Ecco la libreria magica!

int main(int argc, char **argv) {
    if (argc < 2) {
        fprintf(stderr, "Uso: %s <n>\\n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]);
    size_t bytes = n * n * sizeof(float);

    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);

    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0f;
            h_b[i * n + j] = 3.0f;
            h_c[i * n + j] = 0.0f;
        }
    }

    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // --- INIZIO MAGIA CUBLAS ---

    // 1. Creiamo un "handle", ovvero il contesto per la libreria
    cublasHandle_t handle;
    cublasCreate(&handle);

    // cuBLAS calcola un'operazione generica: C = (alpha * A * B) + (beta * C)
    // Noi vogliamo solo C = A * B, quindi impostiamo alpha a 1 e beta a 0
    float alpha = 1.0f;
    float beta = 0.0f;

    // Timing con Eventi CUDA
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    // 2. Lanciamo la funzione Sgemm (Single precision GEneral Matrix Multiply)
    // TRUCCHETTO: cuBLAS usa il formato "Column-Major" (colonne in memoria, stile Fortran).
    // Noi in C usiamo "Row-Major" (righe in memoria). Per fare in modo che la matematica
    // torni perfetta senza dover riordinare a mano le matrici, basta passare a cuBLAS
    // prima la matrice B e poi la matrice A!
    cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N,
                n, n, n,
                &alpha,
                d_b, n,  // <--- Nota: passiamo d_b per primo!
                d_a, n,
                &beta,
                d_c, n);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);


    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("cuBLAS Time for n=%d: %.4f seconds\n", n, milliseconds / 1000.0f);


    // 3. Chiudiamo l'handle
    cublasDestroy(handle);

    // --- FINE MAGIA CUBLAS ---


    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    // Stampa di controllo
    FILE *f = fopen("mat-res.txt", "w");
    if (f) {
        fprintf(f, "%d\\n\\n", n);
        int sample = (n < 100) ? n : 10;
        for (int i = 0; i < sample; i++) {
            for (int j = 0; j < sample; j++) {
                fprintf(f, "%.0f ", h_c[i * n + j]);
            }
            fprintf(f, "\\n");
        }
        fclose(f);
    }

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);

    return 0;
}
'''

with open('cuda_matrixmult_cuBLAS.cu', 'w') as f:
    f.write(cell_str)

In [2]:
!nvcc -arch=sm_75 -O3 -lcublas cuda_matrixmult_cuBLAS.cu -o cuda_matrixmult_cuBLAS

In [3]:
!nvprof ./cuda_matrixmult_cuBLAS 17050

==3928== NVPROF is profiling process 3928, command: ./cuda_matrixmult_cuBLAS 17050
cuBLAS Time for n=17050: 1.8904 seconds
==3928== Profiling application: ./cuda_matrixmult_cuBLAS 17050
==3928== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   70.64%  1.83989s         1  1.83989s  1.83989s  1.83989s  volta_sgemm_128x128_nn
                   19.83%  516.48ms         2  258.24ms  252.73ms  263.74ms  [CUDA memcpy HtoD]
                    9.53%  248.19ms         1  248.19ms  248.19ms  248.19ms  [CUDA memcpy DtoH]
      API calls:   64.36%  1.83989s         1  1.83989s  1.83989s  1.83989s  cudaEventSynchronize
                   26.81%  766.41ms         3  255.47ms  248.60ms  263.97ms  cudaMemcpy
                    6.90%  197.32ms         6  32.887ms  10.774us  196.78ms  cudaMalloc
                    0.85%  24.307ms         1  24.307ms  24.307ms  24.307ms  cudaGetSymbolAddress
                    0.34%  9.8573ms      